In [1]:
import numpy as np
import pandas as pd
from pylab import *
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import pandas as pd
from pylab import *
from sequana import FastA
import tensorflow as tf
from tensorflow.keras import layers, models
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import random
from collections import defaultdict
from sklearn.metrics import classification_report


2025-07-28 13:49:10.099169: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-28 13:49:10.103769: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-28 13:49:10.116279: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753703350.137005  142442 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753703350.143257  142442 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753703350.159638  142442 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
centromeres = pd.read_csv("../../output/estimation/major/Major.csv")
centromeres = {
    str(row['Chromosome']): (row['start'], row['end'])
    for _, row in centromeres.iterrows()
}


In [3]:
f = FastA("../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta")

In [4]:
def one_hot_encoding(x):
    if x == 'A':
        return np.array([1,0,0,0])
    elif x == 'C':
        return np.array([0,1,0,0])
    elif x == 'G':
        return np.array([0,0,1,0])
    elif x == 'T':
        return np.array([0,0,0,1])
    else:
        return np.array([0,0,0,0])
        

In [5]:
SIZE=3000

In [6]:
def get_true_positive(size=3000):
    data = []
    for chrom in range(1,36+1):
        start, stop = centromeres[str(chrom)]
        #seq = f.sequences[f.names.index(str(chrom))]
        seq = f.sequences[chrom-1]

        if stop-start != 3000:
            stop = start + size
        # flip to get more positives
        data.append([one_hot_encoding(x) for x in seq[start:stop]])
        data.append([one_hot_encoding(x) for x in seq[start:stop][::-1]])
        

    return data
positives = get_true_positive(SIZE)

In [7]:
lengths = list(f.get_lengths_as_dict().values())

In [8]:

##################################### WARNING #############################################
############################### REMOVE FALSE NEGATIVE (centromeres) #######################

def get_true_negatives(N=1000,size=3000,seed=42):
    data = []
    positions = defaultdict(list)
    random.seed(seed)

    # get the random combos first
    for i in tqdm(range(N)):
        chrom = random.randint(1,36)
        N = lengths[chrom-1]
        pos = random.randint(1, N-size)
        start, stop = centromeres[str(chrom)]
        if pos>start and pos<stop:
            pass # this is a centromeres so not a negative
        else:
            positions[chrom].append(pos)
        
        
    for chrom in tqdm(positions.keys()):
        seq = f.sequences[chrom-1]
        for position in positions[chrom]:
            data.append([one_hot_encoding(x) for x in seq[position:position+size]])
    return data

    #negatives = get_true_negatives(N=10000, size=SIZE)

In [16]:
kermel_values = [[15,10]]
for kermel in kermel_values:
    #results = []
    f = FastA("../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta")
    for i in range(7,11):
        print("###############################################################################")
        print(f'{i}/10')
        print("###############################################################################")

        negatives = get_true_negatives(N=10000, size=SIZE,seed=i)
     
        
        X_pos = positives
        X_neg = negatives
        
        
        # Create label arrays
        y_pos = [1] * len(X_pos)
        y_neg = [0] * len(X_neg)
        
        # Combine and shuffle
        X = np.array(X_pos + X_neg)  # shape: (N, 3000, 4)
        y = np.array(y_pos + y_neg)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
    
    
        layer_size=16

        model = models.Sequential([
            layers.Input(shape=(SIZE, 4)),
            layers.Conv1D(layer_size, kernel_size=kermel[0], activation='relu'),
            layers.GlobalMaxPooling1D(),
            layers.Dense(1, activation='sigmoid')  # binary classification
        ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        
        seed = 42
    
        tf.random.set_seed(seed)
        
        tf.config.experimental.enable_op_determinism()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=80,
            batch_size=32,
            class_weight={0: 1, 1: len(y_neg)/len(y_pos)},  # handle imbalance
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
        )
    
    
        y_pred = model.predict(X_test) > 0.2
        report = classification_report(y_test, y_pred, output_dict=True)
    
        metrics = {
            'f1_score': report['1']['f1-score'],
            'precision': report['1']['precision'],
            'recall': report['1']['recall']
        }
        results.append(metrics)

    with open(f"input_conv1d_globalmax_dense_result_bis.pkl", "wb") as f:
        pickle.dump(results, f)




###############################################################################
7/10
###############################################################################


100%|███████████████████████████████████████████| 36/36 [00:30<00:00,  1.18it/s]


Epoch 1/80


2025-07-28 15:24:45.623097: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


250/251 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9581 - loss: 1.5602

...

250/251 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9536 - loss: 1.6542

2025-07-28 15:43:56.587782: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


251/251 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.9528 - loss: 1.6541 - val_accuracy: 0.6306 - val_loss: 0.6754
Epoch 2/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.8294 - loss: 1.0577 - val_accuracy: 0.7793 - val_loss: 0.6347
Epoch 3/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.8866 - loss: 0.9660 - val_accuracy: 0.8632 - val_loss: 0.5734
Epoch 4/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9199 - loss: 0.8646 - val_accuracy: 0.9016 - val_loss: 0.5091
Epoch 5/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9400 - loss: 0.7584 - val_accuracy: 0.9241 - val_loss: 0.4392
Epoch 6/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9557 - loss: 0.6537 - val_accuracy: 0.9466 - val_loss: 0.3716
Epoch 7/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9658 - loss: 0.5599 - val_accuracy: 0.9581 - val_loss: 0.3119
Epoch 8/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.9715 - loss: 0.4726 - val_accuracy: 0.96

2025-07-28 15:49:11.912312: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


In [11]:
with open(f"{64}_layer_result_patience5.pkl", "wb") as f:
    pickle.dump(results, f)


In [32]:
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Liste des tailles de couches
layer_sizes = [16, 32, 64, 128]

# Charger les résultats et les mettre dans une liste
all_data = []

for layer in layer_sizes: 
    with open(f"{layer}_layer_result.pkl", "rb") as f:
        results = pickle.load(f)
        for metrics in results:
            all_data.append({
                'layer_size': layer,
                'f1_score': metrics['f1_score'],
                'precision': metrics['precision'],
                'recall': metrics['recall']
            })

# Convertir en DataFrame
df = pd.DataFrame(all_data)

# Afficher les 3 boxplots
metrics = ['f1_score', 'precision', 'recall']
for metric in metrics:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='layer_size', y=metric, data=df)
    plt.title(f"Boxplot of {metric} by Layer Size")
    plt.xlabel("Layer Size")
    plt.ylabel(metric.capitalize())
    plt.grid(True)
    plt.tight_layout()
    plt.show()


(8011, 2003)